# Prism Skeptic Test: Is It Data Leakage?

The concern: extracting spectra from a model trained on Shakespeare, then
initializing a model to train on Shakespeare — are we just leaking task info?

Three tests:
1. **Same-data** (control): extract from Shakespeare, train on Shakespeare (what we've been doing)
2. **Cross-data**: extract from a model trained on **different text**, train on Shakespeare
3. **Wrong-data**: extract from a model trained on **random/shuffled** data, train on Shakespeare

If cross-data helps almost as much as same-data → spectral structure is architectural, not task-specific.
If cross-data doesn't help → we were leaking task info. The advantage is real but narrow.

We split Shakespeare 50/50 into two non-overlapping halves for the cross-data test.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py 2>/dev/null
print('Ready.')

In [ ]:
# Cell 2: Create split datasets + shuffled dataset
import numpy as np
import os

# Load the full Shakespeare data
data = np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/shakespeare_char/val.bin', dtype=np.uint16, mode='r')
n = len(data)
print(f'Full Shakespeare: {n:,} tokens train, {len(val_data):,} tokens val')

# Split A: first half for teacher training
# Split B: second half for student training
half = n // 2
split_a = np.array(data[:half], dtype=np.uint16)
split_b = np.array(data[half:], dtype=np.uint16)

# Shuffled: same tokens but randomly reordered (destroys all structure)
shuffled = np.array(data[:half], dtype=np.uint16).copy()
np.random.seed(42)
np.random.shuffle(shuffled)

# Save split datasets
for name, train_data in [('split_a', split_a), ('split_b', split_b), ('shuffled', shuffled)]:
    d = f'data/shakespeare_{name}'
    os.makedirs(d, exist_ok=True)
    # Train on this split, validate on the standard val set
    train_data.tofile(os.path.join(d, 'train.bin'))
    val_data.tofile(os.path.join(d, 'val.bin'))
    # Copy meta.pkl for vocab
    import shutil
    shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d, 'meta.pkl'))
    print(f'  {name}: {len(train_data):,} train tokens')

print('Datasets ready.')

In [ ]:
# Cell 3: Train three teacher models and extract spectra
import subprocess, os

teachers = [
    ('same', 'shakespeare_char', 'Teacher on full Shakespeare (same data)'),
    ('cross', 'shakespeare_split_a', 'Teacher on first half (different data)'),
    ('random', 'shakespeare_shuffled', 'Teacher on shuffled tokens (no structure)'),
]

for tag, dataset, desc in teachers:
    cache = f'.prism_cache/{tag}'
    if os.path.exists(f'{cache}/spectra.json'):
        print(f'  [{tag}] Cached.')
        continue
    
    print(f'\n  Training: {desc}')
    result = subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        f'--dataset={dataset}',
        '--max_iters=2000', '--eval_interval=2000', '--eval_iters=50',
        '--log_interval=1000', f'--out_dir=out-teacher-{tag}',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, text=True, timeout=600)
    
    # Check if it trained
    import re
    for line in result.stdout.split('\n'):
        m = re.search(r'step \d+: train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            print(f'    Final: train={m.group(1)}, val={m.group(2)}')
    
    if result.returncode != 0:
        print(f'    ERROR: {result.stderr[-300:]}')
        continue
    
    print(f'  Extracting spectra...')
    subprocess.run([
        'python', 'prism_extract.py',
        '--ckpt', f'out-teacher-{tag}/ckpt.pt',
        '--out', cache,
    ], capture_output=True, timeout=120)
    print(f'  Saved to {cache}/')

print('\nAll teachers trained and extracted.')

In [ ]:
# Cell 4: Run the skeptic test — all train on split_b (second half)
# This way even the "same" condition uses spectra from full data
# but trains on only half — partial overlap, not full leak.
import subprocess, time, re, json, os

STEPS = 5000
EVAL = 100
TRAIN_DATA = 'shakespeare_split_b'  # everyone trains on this

def prism_args(tag):
    return [
        '--prism_init=True', '--prism_align=0.75',
        f'--prism_spectra=.prism_cache/{tag}/spectra.json',
        f'--prism_directions=.prism_cache/{tag}/directions.pt',
    ]

configs = [
    ('baseline', ['--prism_init=False']),
    ('same_data_marathon', prism_args('same') + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    ('cross_data_marathon', prism_args('cross') + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    ('random_data_marathon', prism_args('random') + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999']),
    ('same_data_sprint', prism_args('same') + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999']),
    ('cross_data_sprint', prism_args('cross') + [
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.005', '--prism_mod_decay=0.999']),
]

all_curves = {}

for name, extra in configs:
    log_path = f'/content/skeptic_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'\n{"="*60}')
        print(f'  {name}')
        print(f'{"="*60}')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             f'--dataset={TRAIN_DATA}',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=250',
             f'--out_dir=out-skeptic-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'ERROR: {result.stderr[-500:]}')
        print(f'  Wall: {time.time()-t0:.0f}s')

    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val:
        best_v = min(val.values())
        best_s = min(val, key=val.get)
        print(f'  [{name}] best: {best_v:.4f} @ step {best_s}')

with open('/content/skeptic_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 5: Verdict
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/skeptic_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*70)
print('  SKEPTIC TEST: Is Prism data leakage?')
print('='*70)
print(f'  All models train on Shakespeare split B (second half).')
print(f'  Baseline best: {bb:.4f} at step {bs}')

print(f'\n{"Config":>25s}  {"Spectra from":>15s}  {"Best val":>8s}  {"@step":>6s}  {"@5000":>8s}')
print('-' * 68)
labels = {
    'baseline': 'none',
    'same_data_marathon': 'full shakes',
    'cross_data_marathon': 'first half',
    'random_data_marathon': 'shuffled',
    'same_data_sprint': 'full shakes',
    'cross_data_sprint': 'first half',
}
for name in ['baseline', 'same_data_marathon', 'cross_data_marathon', 'random_data_marathon',
             'same_data_sprint', 'cross_data_sprint']:
    c = curves.get(name, {})
    if not c: continue
    best_v = min(c.values())
    best_s = min(c, key=c.get)
    at_5k = c.get(5000, c.get(max(c.keys()), 0))
    print(f'{name:>25s}  {labels.get(name,"?"):>15s}  {best_v:>8.4f}  {best_s:>6d}  {at_5k:>8.4f}')

print(f'\n--- Interpretation ---')
sm = curves.get('same_data_marathon', {})
cm = curves.get('cross_data_marathon', {})
rm = curves.get('random_data_marathon', {})
if sm and cm and rm:
    sm_best = min(sm.values())
    cm_best = min(cm.values())
    rm_best = min(rm.values())
    print(f'  Same-data marathon:   {sm_best:.4f}')
    print(f'  Cross-data marathon:  {cm_best:.4f}')
    print(f'  Random-data marathon: {rm_best:.4f}')
    print(f'  Baseline:             {bb:.4f}')
    if cm_best < bb - 0.01:
        pct = (sm_best - bb) / (cm_best - bb) * 100 if cm_best != bb else 0
        print(f'\n  Cross-data retains {pct:.0f}% of same-data advantage.')
        print(f'  → Spectral structure is partially ARCHITECTURAL, not just task leakage.')
    elif cm_best < bb:
        print(f'\n  Cross-data helps slightly. Mostly architectural signal.')
    else:
        print(f'\n  Cross-data does NOT help. The advantage is task-specific data leakage.')
    if rm_best >= bb - 0.005:
        print(f'  Shuffled-data does NOT help → spectral structure requires real language, not just token stats.')

In [ ]:
# Cell 6: Download
from google.colab import files
files.download('/content/skeptic_curves.json')
for n in ['baseline','same_data_marathon','cross_data_marathon','random_data_marathon','same_data_sprint','cross_data_sprint']:
    try: files.download(f'/content/skeptic_{n}.txt')
    except: pass